# Growth-Spurt — a quantitative teardown 🔬
### The long-short on real EDGAR data · the backwards sign · micro-caps, the investment factor, survivorship

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Sharpe 0.835 replicates?: Busted](https://img.shields.io/badge/Sharpe_0.835_replicates%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We rebuild the asset-growth long-short on SEC balance-sheet data for large caps and explain why the highest headline Sharpe on the list doesn't replicate.

> ⚠️ **Not investment advice.** EDGAR `Assets` (10-K) + Yahoo annual returns, ~399 current S&P 500 members (survivorship-biased, large-cap — both stated), 2005–2025. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (growth_spurt/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from growth_spurt import data, strategy as st
ag, fwd = data.fetch_panel()                  # cache-first; built by examples/verify.py --fetch
h = st.quantile_hedge(ag, fwd, q=0.2)
mkt = st.market_annual(fwd).reindex(h.index)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | hedge −1.6%/yr, Sharpe −0.12, sign reversed |
| Tradability | **Mirage** | 0.835 lives in micro-caps, not large caps |
| Replicates? | **Busted** | even a survivorship-flattered test can't find it |

> 💡 *In plain words:* the most attractive number on the list is the least tradable.

## 1 · The claim, steelmanned

- **H₁:** low-asset-growth firms out-earn high-growth ones (positive hedge).
- **H₂ (the pitch):** the hedge delivers a Sharpe near the vendor's 0.835 on a tradable universe.
- **H₃:** the effect is a standalone anomaly, not a known factor.

## 2 · So what? — what rides on each

If H₂ holds, an annual balance-sheet sort beats the market handsomely. If it fails on large caps, the headline is a micro-cap artifact.

## 3 · How we'd know — the protocol

EDGAR `Assets` → fiscal-year asset growth → annual long-bottom-20% / short-top-20% → hedge vs equal-weight universe → inspect the leg means and the sign → weigh the three standard explanations.

## 4 · The teardown

### 4.1 The long-short, by year

In [2]:
display(h.round(3))
s=st.summary(h['hedge']); print(f"hedge: mean {s['mean']:+.2%}/yr  Sharpe {s['sharpe']:.2f}  hit {s['hit_rate']:.0%}  n={s['n']}")

,low_growth,high_growth,hedge
2009,0.223,0.116,0.107
2010,0.031,0.043,-0.012
2011,0.291,0.282,0.010
2012,0.532,0.403,0.129
2013,0.180,0.253,-0.073
2014,-0.020,0.122,-0.142
2015,0.265,0.127,0.138
2016,0.273,0.356,-0.083
2017,-0.058,0.026,-0.084
2018,0.338,0.413,-0.075


hedge: mean -1.62%/yr  Sharpe -0.12  hit 47%  n=17


> 💡 *In plain words:* a negative mean and a sub-50% hit rate. **H₁ rejected** on large caps.

### 4.2 The backwards sign

In [3]:
print('long  (low growth) mean: %+.2f%%' % (st.summary(h['low_growth'])['mean']*100))
print('short (high growth) mean: %+.2f%%' % (st.summary(h['high_growth'])['mean']*100))
print('vendor headline Sharpe: 0.835  |  our hedge Sharpe: %.2f' % st.summary(h['hedge'])['sharpe'])

long  (low growth) mean: +19.75%
short (high growth) mean: +21.37%
vendor headline Sharpe: 0.835  |  our hedge Sharpe: -0.12


> 💡 *In plain words:* high-growth firms *beat* low-growth ones here — the opposite of the anomaly. **H₂ rejected.**

### 4.3 Why it fails — three standard reasons

1. **Micro-cap concentration.** Hou-Xue-Zhang (2020) show accounting anomalies, asset growth included, concentrate in the smallest names; a large-cap universe is the wrong place.
2. **The investment factor.** In Fama-French five-factor, asset growth *is* the CMA factor — already priced, not a free anomaly. **H₃ rejected.**
3. **Survivorship flatters the wrong way.** Current members exclude delisted growth blow-ups that would *help* the short — so even this biased test, which should favour the strategy, still finds nothing.

## 5 · The verdict

H₁, H₂, H₃ all rejected on tradable names → Signal `NONE`, Tradability `MIRAGE`, headline `BUSTED`.

## 6 · Could you trade it?

Only by shorting hundreds of micro-caps — illiquid, hard-to-borrow, survivorship- and data-fragile. The liquid version is flat-to-negative. The 0.835 is a property of the untradable tail.

## 7 · Going further

Forks: (a) add a small/mid-cap ETF-of-stocks universe to see the effect strengthen toward the micro-cap end; (b) residualise asset growth against the CMA factor — does anything remain? (c) point-in-time membership to kill the survivorship flatter. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).